# DriftWise: real-data gate ablation (UNSW-NB15)
Does the sequential gate beat simple 'switch if it looks better' rules on the real stream?
Also reports what each accepted update *really* did to accuracy on the next 10,000 rows (hindsight).

**How to run:**
1. Run cell 1 (install). Then **Runtime > Restart session**.
2. Do NOT run cell 1 again. Run cell 2 (setup; allow Google Drive access when asked), then cell 3.
3. Cell 3 takes about 15 to 25 minutes. Keep the tab open. Paste the two printed tables back into the chat.

Needs `UNSW_NB15_DriftWise_clean_chronological.csv.gz` in `MyDrive/DriftWise/data/processed/` and `ega_gate.py` in `MyDrive/DriftWise/src/`.

## 1. Install (run once, then Runtime > Restart session)

In [ ]:
!pip install river "numpy<2.3" -q

## 2. Setup: load the real data and build features

In [ ]:
import os, sys
import pandas as pd, numpy as np
from google.colab import drive
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import recall_score, balanced_accuracy_score
drive.mount("/content/drive")

path = "/content/drive/MyDrive/DriftWise/data/processed/UNSW_NB15_DriftWise_clean_chronological.csv.gz"
df = pd.read_csv(path, low_memory=False)
for c in ["sport", "dsport"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

drop_cols = ["Label", "attack_cat", "srcip", "dstip", "Stime", "Ltime", "Stime_UTC", "Ltime_UTC"]
cat_cols = ["proto", "state", "service"]
num_cols = [c for c in df.columns if c not in drop_cols + cat_cols]

train_end = 17344                     # the first slice of the active window
train = df.iloc[:train_end]
keep = {c: set(train[c].value_counts().head(10 if c == "proto" else 50).index) for c in cat_cols}

def make_features(d):
    num = d[num_cols].fillna(-1).astype(float)
    num = np.sign(num) * np.log1p(np.abs(num))            # tame very large values
    cats = d[cat_cols].copy()
    for c in cat_cols:
        cats[c] = cats[c].where(cats[c].isin(keep[c]), "other")
    return pd.concat([num, pd.get_dummies(cats, dtype=float)], axis=1)

last_attack = int(np.where(df["Label"].values == 1)[0].max())
N = last_attack + 1                                   # the active window
columns = make_features(train).columns
X_raw = make_features(df.iloc[:N]).reindex(columns=columns, fill_value=0.0).to_numpy()
y_all = df["Label"].to_numpy()[:N]
print("Active window:", N, "rows,", X_raw.shape[1], "feature columns")

SEED = 42
FITS = {"n": 0}
def fit_on(a, b):
    FITS["n"] += 1
    sc = StandardScaler().fit(X_raw[a:b])
    m = SGDClassifier(loss="log_loss", class_weight="balanced", random_state=SEED)
    m.fit(sc.transform(X_raw[a:b]), y_all[a:b])
    return sc, m

def run_periodic(period=None, window=20000, chunk=1000):
    """Retrain every `period` rows on the newest `window` rows (labels assumed to arrive immediately)."""
    sc, m = fit_on(0, train_end)
    preds_p = np.zeros(N, dtype=int)
    n_updates = 0
    for s in range(train_end, N, chunk):
        e = min(s + chunk, N)
        if period and s > train_end and (s - train_end) % period == 0:
            sc, m = fit_on(max(0, s - window), s)
            n_updates += 1
        preds_p[s:e] = m.predict(sc.transform(X_raw[s:e]))
    return preds_p, n_updates

def score(p_all, lo, hi):
    yy, pp = y_all[lo:hi], p_all[lo:hi]
    return (round(balanced_accuracy_score(yy, pp), 3), round(recall_score(yy, pp, pos_label=1), 3),
            round(recall_score(yy, pp, pos_label=0), 3))
print("Setup done.")

## 3. Ablation: sequential gate vs simple margin rules on the real stream

In [ ]:
import sys, time
import numpy as np, pandas as pd
from river import drift
sys.path.insert(0, "/content/drive/MyDrive/DriftWise/src")
from ega_gate import sequential_gate as seq_gate

def bal(y, p):
    r = [float((p[y == c] == c).mean()) for c in (0, 1)]
    return float(np.mean(r)), float(min(r))

def make_seq(A):
    """Sequential gate with total significance level A (0.05 = the setting used so far)."""
    def gate(y, p_new, p_old, look=None, alpha_total=0.05, **k):
        return seq_gate(y, p_new, p_old, look=look, alpha_total=alpha_total * (A / 0.05))
    return gate

def make_point(margin, min_n=200, min_per_class=30):
    """Simple switch: accept if the candidate's balanced accuracy beats the current model's by more than `margin`."""
    def gate(y, p_new, p_old, look=None, alpha_total=None, **k):
        y, p_new, p_old = map(np.asarray, (y, p_new, p_old))
        n1 = int(y.sum()); n0 = len(y) - n1
        if len(y) < min_n or n1 < min_per_class or n0 < min_per_class:
            return {"decision": "DEFER", "est_balacc": 0.0}
        diff = bal(y, p_new)[0] - bal(y, p_old)[0]
        return {"decision": "ACCEPT" if diff > margin else "REJECT", "est_balacc": diff}
    return gate

GATES = {"sequential gate (alpha=0.05)": make_seq(0.05),
         "point rule, margin=0": make_point(0.0),
         "point rule, margin=0.01": make_point(0.01),
         "point rule, margin=0.02": make_point(0.02)}

class AlarmMonitor:
    """Two ADWIN detectors: one on missed attacks, one on false alarms. Only UP alarms (errors rising) count."""
    def __init__(self):
        self.reset()
    def reset(self):
        self.det = {1: drift.ADWIN(delta=0.002), 0: drift.ADWIN(delta=0.002)}
        self.hist = {1: [], 0: []}
        self.win = {1: 300, 0: 3000}
    def update(self, yb, eb):
        up = False
        for cls in (1, 0):
            for e in eb[yb == cls]:
                h = self.hist[cls]
                h.append(int(e))
                out = self.det[cls].update(int(e))
                detected = out[0] if isinstance(out, tuple) else self.det[cls].drift_detected
                w = self.win[cls]
                if detected and len(h) >= 2 * w:
                    if np.mean(h[-w:]) > np.mean(h[-2 * w:-w]):
                        up = True
        return up

def run_gate(gate, val_size=3000, gap=500, short=10000, long=30000,
             gate_every=1000, T=15000, patience=3, chunk=1000):
    """EGA v2 on the real stream with a chosen accept/reject rule. Labels arrive immediately."""
    FITS["n"] = 0
    sc, m = fit_on(0, train_end)
    preds_r = np.zeros(N, dtype=int)
    mon = AlarmMonitor()
    in_ep, alarm_t, look, n_acc, streak = False, None, 0, 0, 0
    deployed = []                                        # (time, old model, new model) for every accepted update
    for s in range(train_end, N, chunk):
        e = min(s + chunk, N)
        pb = m.predict(sc.transform(X_raw[s:e]))
        preds_r[s:e] = pb
        t = e                                            # labels are known for rows before t
        fired = mon.update(y_all[s:e], (pb != y_all[s:e]).astype(int))
        if (not in_ep) and fired:
            in_ep, alarm_t, look, n_acc, streak = True, t, 0, 0, 0
        if in_ep and (t - alarm_t) % gate_every == 0:
            look += 1
            te = t - val_size - gap
            Xv, yv = X_raw[t - val_size:t], y_all[t - val_size:t]
            p_cur = m.predict(sc.transform(Xv))
            best = None
            for w in (short, long):
                c_sc, c_m = fit_on(max(0, te - w), te)
                r = gate(yv, c_m.predict(c_sc.transform(Xv)), p_cur, look=look, alpha_total=0.05 / 2)
                if r["decision"] == "ACCEPT" and (best is None or r["est_balacc"] > best[2]["est_balacc"]):
                    best = (c_sc, c_m, r)
            if best is not None:
                deployed.append((t, sc, m, best[0], best[1]))
                sc, m = best[0], best[1]
                n_acc += 1; streak = 0
            else:
                streak += 1
                if n_acc > 0 and streak >= patience:
                    in_ep = False; mon.reset()
                elif t - alarm_t >= T:
                    in_ep = False; mon.reset()
            if in_ep and t - alarm_t >= T:
                in_ep = False; mon.reset()
    return preds_r, deployed, FITS["n"]

def realized_benefit(deployed, horizon=10000, min_future=2000):
    """For each accepted update: balanced accuracy of the new model minus the old one on the NEXT 10,000 rows
    (rows the gate had not seen when it decided). This is the update's real effect, in hindsight."""
    diffs = []
    for t, osc, om, nsc, nm in deployed:
        hi = min(N, t + horizon)
        yy = y_all[t:hi]
        if hi - t < min_future or yy.sum() < 30:
            continue
        p_old = om.predict(osc.transform(X_raw[t:hi]))
        p_new = nm.predict(nsc.transform(X_raw[t:hi]))
        diffs.append(bal(yy, p_new)[0] - bal(yy, p_old)[0])
    return diffs

SEEDS = [42, 7]
rows, benefit = [], []
for SEED in SEEDS:
    print("random seed", SEED, "...")
    t0 = time.time()
    p, nu = run_periodic(period=None)
    a1, a2 = score(p, train_end, N), score(p, 117000, N)
    rows.append({"seed": SEED, "method": "never update", "updates": 0, "models trained": 1,
                 "ALL bal_acc": a1[0], "DRIFT bal_acc": a2[0], "DRIFT attack_recall": a2[1]})
    FITS["n"] = 0
    p, nu = run_periodic(period=10000, window=50000)
    a1, a2 = score(p, train_end, N), score(p, 117000, N)
    rows.append({"seed": SEED, "method": "periodic (every 10,000, newest 50,000)", "updates": nu, "models trained": FITS["n"],
                 "ALL bal_acc": a1[0], "DRIFT bal_acc": a2[0], "DRIFT attack_recall": a2[1]})
    for name, g in GATES.items():
        p, dep, fits = run_gate(g)
        a1, a2 = score(p, train_end, N), score(p, 117000, N)
        rows.append({"seed": SEED, "method": "EGA v2: " + name, "updates": len(dep), "models trained": fits,
                     "ALL bal_acc": a1[0], "DRIFT bal_acc": a2[0], "DRIFT attack_recall": a2[1]})
        for d in realized_benefit(dep):
            benefit.append({"seed": SEED, "method": "EGA v2: " + name, "realized_change": d})
        print(f"   done: {name}  ({time.time() - t0:.0f}s so far)")
R = pd.DataFrame(rows)
Bn = pd.DataFrame(benefit)

pd.set_option("display.width", 300, "display.max_columns", 20, "display.max_colwidth", None)
order = list(dict.fromkeys(R["method"]))
print("\nMEAN OVER SEEDS", SEEDS)
display(R.groupby("method")[["updates", "models trained", "ALL bal_acc", "DRIFT bal_acc", "DRIFT attack_recall"]]
        .mean().round(3).loc[order])

print("\nWHAT EACH ACCEPTED UPDATE REALLY DID (balanced accuracy on the next 10,000 rows, new minus old model):")
tab = []
for name in order:
    if not name.startswith("EGA"):
        continue
    d = Bn[Bn.method == name]["realized_change"]
    tab.append({"method": name, "updates judged": len(d),
                "mean change": round(d.mean(), 4) if len(d) else None,
                "harmful (< -0.01)": int((d < -0.01).sum()),
                "helpful (> +0.01)": int((d > 0.01).sum())})
display(pd.DataFrame(tab).set_index("method"))

os.makedirs("/content/drive/MyDrive/DriftWise/results", exist_ok=True)
R.to_csv("/content/drive/MyDrive/DriftWise/results/real_data_gate_ablation.csv", index=False)
Bn.to_csv("/content/drive/MyDrive/DriftWise/results/real_data_gate_ablation_update_effects.csv", index=False)
print("Saved: real_data_gate_ablation.csv and real_data_gate_ablation_update_effects.csv")